# European Aviation & Airport Operations Analysis

This notebook processes, harmonizes, and analyzes European airport data from multiple sources:
1. **Eurostat Passenger Traffic**: Monthly passenger flows across European airports.
2. **OurAirports Data**: Airport metadata and geographic identifiers.
3. **European Air Routes**: Flight routes network and scheduling.
4. **Airport Operations & Movements**: Detailed monthly flight totals, arrivals, and departures.

## 1. Setup & Environment

In [9]:
import pandas as pd
import numpy as np

## 1. Airport Passenger Traffic (Eurostat)
Ingest Eurostat monthly passenger dataset, parse composite identifiers, compute monthly averages and annual estimates, and enrich with airport metadata for Schengen countries.

In [10]:
# Load Eurostat dataset
traffic = pd.read_csv("./estat_ttr00017 (1).tsv", sep="\t")

# Handle missing value flags represented by ":"
traffic = traffic.replace(":", np.nan)
traffic = traffic.dropna(how="all")

# Parse airport ICAO code and country code from composite index column
traffic["airport"] = traffic.iloc[:, 0].str[-4:]
traffic["country"] = traffic.iloc[:, 0].str.split("_").str[-2].str[-2:]

traffic.head(10)

,"freq,unit,schedule,tra_cov,tra_meas,rep_airp\TIME_PERIOD",2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,airport,country
0,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWG",68694,80716,83376,85487,81717,69465,53148,47527,39069,44339,51258,58083,68024,77423,87068,:,LOWG,AT
1,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWI",18567,27352,28409,30382,25516,9628,6399,98024,178197,174389,169886,34786,19259,27507,30517,:,LOWI,AT
2,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWK",15736,15635,15542,18418,14606,12048,1816,3118,2296,2877,2503,9743,13522,14739,16372,:,LOWK,AT
3,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWL",30179,35168,37864,42130,37509,20602,7761,6757,6011,8018,7615,16033,27152,29008,32207,:,LOWL,AT
4,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWS",120936,138522,158398,172119,144375,124710,90961,160512,213776,212107,179163,102317,126497,141963,171527,:,LOWS,AT
5,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWW",2834855,2962889,3246354,3370748,3125593,3040639,2366088,2489849,1860234,1884763,2225144,2552428,2724503,2820256,3120602,:,LOWW,AT
6,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBAW",15711,16587,28164,29388,28879,24503,11157,11159,11327,10719,:,:,:,:,:,:,EBAW,BE
7,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBBR",1991341,2129320,2503922,2484301,2269505,2220912,1648034,1777476,1567207,1618779,:,:,:,:,:,:,EBBR,BE
8,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBCI",1056388,976934,1060198,1110556,1005260,1035044,838076,899331,739181,809020,:,:,:,:,:,:,EBCI,BE
9,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBLG",6346,118,18840,18348,15994,11730,8299,7326,2417,933,:,:,:,:,:,:,EBLG,BE


In [11]:
# Strip any spaces from column headers
traffic.columns = traffic.columns.str.strip()

# Target monthly periods
columns_dates = ["2025-05", "2025-06", "2025-07", "2025-08", "2025-09", "2025-10", 
                 "2025-11", "2025-12", "2026-01", "2026-02","2026-03", "2026-04"]

# Convert monthly traffic metrics to numeric
traffic[columns_dates] = traffic[columns_dates].apply(pd.to_numeric, errors="coerce")

# Calculate monthly mean (in thousands of passengers) and estimated annual total
traffic["mean"] = traffic[columns_dates].mean(axis=1) / 1000
traffic["total"] = traffic["mean"] * 12

traffic.head(10)

,"freq,unit,schedule,tra_cov,tra_meas,rep_airp\TIME_PERIOD",2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,...,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,airport,country,mean,total
0,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWG",68694.0,80716.0,83376.0,85487.0,81717.0,69465.0,53148.0,47527.0,39069.0,...,51258.0,58083.0,68024,77423,87068,:,LOWG,AT,63.573250,762.8790
1,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWI",18567.0,27352.0,28409.0,30382.0,25516.0,9628.0,6399.0,98024.0,178197.0,...,169886.0,34786.0,19259,27507,30517,:,LOWI,AT,66.794583,801.5350
2,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWK",15736.0,15635.0,15542.0,18418.0,14606.0,12048.0,1816.0,3118.0,2296.0,...,2503.0,9743.0,13522,14739,16372,:,LOWK,AT,9.528167,114.3380
3,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWL",30179.0,35168.0,37864.0,42130.0,37509.0,20602.0,7761.0,6757.0,6011.0,...,7615.0,16033.0,27152,29008,32207,:,LOWL,AT,21.303917,255.6470
4,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWS",120936.0,138522.0,158398.0,172119.0,144375.0,124710.0,90961.0,160512.0,213776.0,...,179163.0,102317.0,126497,141963,171527,:,LOWS,AT,151.491333,1817.8960
5,"M,PAS,TOTAL,INTL,PAS_CRD,AT_LOWW",2834855.0,2962889.0,3246354.0,3370748.0,3125593.0,3040639.0,2366088.0,2489849.0,1860234.0,...,2225144.0,2552428.0,2724503,2820256,3120602,:,LOWW,AT,2663.298667,31959.5840
6,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBAW",15711.0,16587.0,28164.0,29388.0,28879.0,24503.0,11157.0,11159.0,11327.0,...,NaN,NaN,:,:,:,:,EBAW,BE,18.759400,225.1128
7,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBBR",1991341.0,2129320.0,2503922.0,2484301.0,2269505.0,2220912.0,1648034.0,1777476.0,1567207.0,...,NaN,NaN,:,:,:,:,EBBR,BE,2021.079700,24252.9564
8,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBCI",1056388.0,976934.0,1060198.0,1110556.0,1005260.0,1035044.0,838076.0,899331.0,739181.0,...,NaN,NaN,:,:,:,:,EBCI,BE,952.998800,11435.9856
9,"M,PAS,TOTAL,INTL,PAS_CRD,BE_EBLG",6346.0,118.0,18840.0,18348.0,15994.0,11730.0,8299.0,7326.0,2417.0,...,NaN,NaN,:,:,:,:,EBLG,BE,9.035100,108.4212


In [12]:
schengen_codes = ["AT", "BE", "BG", "CH", "CZ", "DE", "DK", "EE", "EL", "ES",
                   "FI", "FR", "HR", "HU", "IS", "IT", "LT", "LU", "LV",
                   "MT", "NL", "NO", "PL", "PT", "RO", "SE", "SI", "SK"]

countries = {
    "AT": "Austria", "BE": "Belgium", "BG": "Bulgaria", "CH": "Switzerland",
    "CY": "Cyprus", "CZ": "Czechia", "DE": "Germany", "DK": "Denmark",
    "EE": "Estonia", "EL": "Greece", "ES": "Spain", "FI": "Finland",
    "FR": "France", "HR": "Croatia", "HU": "Hungary", "IE": "Ireland",
    "IS": "Iceland", "IT": "Italy", "LT": "Lithuania", "LU": "Luxembourg",
    "LV": "Latvia", "ME": "Montenegro", "MK": "North Macedonia", "MT": "Malta",
    "NL": "Netherlands", "NO": "Norway", "PL": "Poland", "PT": "Portugal",
    "RO": "Romania", "RS": "Serbia", "SE": "Sweden", "SI": "Slovenia",
    "SK": "Slovakia", "TR": "Turkey"}

# Filter for Schengen members and map ISO codes to country names
traffic = traffic[traffic["country"].isin(schengen_codes)]
traffic["country"] = traffic["country"].map(countries)

# Remove duplicate airports by keeping the highest total traffic record
traffic = traffic.sort_values("total", ascending=False).drop_duplicates(subset="airport", keep="first")

# Select relevant analytical columns
traffic = traffic[["airport", "country", "mean", "total"]].reset_index(drop=True)
traffic.head(10)

,airport,country,mean,total
0,LFPG,France,6038.785900,72465.430800
1,LEMD,Spain,5736.871000,68842.452000
2,EHAM,Netherlands,5705.252500,68463.030000
3,EDDF,Germany,5234.043500,62808.522000
4,LEBL,Spain,4815.955818,57791.469818
5,LIRF,Italy,4238.210917,50858.531000
6,EDDM,Germany,3591.211250,43094.535000
7,LPPT,Portugal,3030.951700,36371.420400
8,LFPO,France,2952.363900,35428.366800
9,LGAV,Greece,2859.674727,34316.096727


## 2. Airport Identification & Metadata (OurAirports)

In [13]:
# Load OurAirports data
airports = pd.read_csv("./airports.csv")

# Merge airport details (name and municipality)
traffic = traffic.merge(airports[["ident", "name", "municipality"]], left_on="airport", right_on="ident",how="left").drop(columns=["ident"])

# Export clean passenger dataset
traffic.to_csv("clean_passengers.csv", index=False)
traffic.head(10)

,airport,country,mean,total,name,municipality
0,LFPG,France,6038.785900,72465.430800,Charles de Gaulle International Airport,"Paris (Roissy-en-France, Val-d'Oise)"
1,LEMD,Spain,5736.871000,68842.452000,Adolfo Suárez Madrid–Barajas Airport,Madrid
2,EHAM,Netherlands,5705.252500,68463.030000,Amsterdam Airport Schiphol,Amsterdam
3,EDDF,Germany,5234.043500,62808.522000,Frankfurt Main Airport,Frankfurt am Main
4,LEBL,Spain,4815.955818,57791.469818,Josep Tarradellas Barcelona-El Prat Airport,Barcelona
5,LIRF,Italy,4238.210917,50858.531000,Rome–Fiumicino Leonardo da Vinci International...,Rome
6,EDDM,Germany,3591.211250,43094.535000,Munich Airport,Munich
7,LPPT,Portugal,3030.951700,36371.420400,Lisbon Humberto Delgado Airport,Lisbon
8,LFPO,France,2952.363900,35428.366800,Paris-Orly Airport,"Paris (Orly, Val-de-Marne)"
9,LGAV,Greece,2859.674727,34316.096727,Athens Eleftherios Venizelos International Air...,Spata-Artemida


## 3. Flight Routes Network Analysis

Clean European route network data, filter routes departing from Schengen countries, and enrich each route with estimated passenger traffic from the departure hub.

In [16]:
# Load European routes dataset
routes = pd.read_csv("./europe_air_routes.csv")

schengen_countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Czech Republic",
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Italy", "Latvia", "Liechtenstein",
    "Lithuania", "Luxembourg", "Malta", "Netherlands", "Norway",
    "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
    "Spain", "Sweden", "Switzerland"]

# Filter routes departing from Schengen countries
routes = routes[routes["departure_country"].isin(schengen_countries)].copy()
routes

,id,iata_from,iata_to,price,day1,day2,day3,day4,day5,day6,...,departure_city,departure_country,departure_ICAO,departure_latitude,departure_longitude,departure_altitude,arrival_ICAO,arrival_latitude,arrival_longitude,arrival_altitude
15,18135,HAJ,AYT,110,yes,yes,yes,yes,yes,yes,...,Hannover,Germany,EDDV,52.461102,9.68508,183,LTAI,36.898701,30.800501,177.0
16,18176,HAJ,MUC,140,yes,yes,yes,yes,yes,yes,...,Hannover,Germany,EDDV,52.461102,9.68508,183,EDDM,48.353802,11.786100,1487.0
17,18157,HAJ,FRA,140,yes,yes,yes,yes,yes,yes,...,Hannover,Germany,EDDV,52.461102,9.68508,183,EDDF,50.033333,8.570556,364.0
18,18132,HAJ,AMS,150,yes,yes,yes,yes,yes,yes,...,Hannover,Germany,EDDV,52.461102,9.68508,183,EHAM,52.308601,4.763890,-11.0
19,18197,HAJ,VIE,160,yes,yes,yes,yes,yes,yes,...,Hannover,Germany,EDDV,52.461102,9.68508,183,LOWW,48.110298,16.569700,600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18106,60483,RMU,BOH,90,no,upcoming,upcoming,no,no,no,...,Murcia,Spain,LEMI,37.803000,-1.12500,632,EGHH,50.779999,-1.842500,38.0
18107,92770,RMU,TFN,220,no,no,no,yes,no,no,...,Murcia,Spain,LEMI,37.803000,-1.12500,632,GCXO,28.482700,-16.341499,2076.0
18108,67738,RMU,OST,190,no,no,no,no,no,no,...,Murcia,Spain,LEMI,37.803000,-1.12500,632,EBOS,51.198898,2.862220,13.0
18109,60480,RMU,ANR,250,no,no,no,no,no,no,...,Murcia,Spain,LEMI,37.803000,-1.12500,632,EBAW,51.189400,4.460280,39.0


In [17]:
# Map annual passenger traffic by departure airport ICAO code
traffic_map = traffic.set_index("airport")["total"]
routes["departure_total_passengers"] = routes["departure_ICAO"].map(traffic_map)

# Export clean routes dataset
routes.to_csv("clean_routes.csv", index=False)
routes.head(10)

,id,iata_from,iata_to,price,day1,day2,day3,day4,day5,day6,...,departure_country,departure_ICAO,departure_latitude,departure_longitude,departure_altitude,arrival_ICAO,arrival_latitude,arrival_longitude,arrival_altitude,departure_total_passengers
15,18135,HAJ,AYT,110,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,LTAI,36.898701,30.800501,177.0,5386.865
16,18176,HAJ,MUC,140,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,EDDM,48.353802,11.786100,1487.0,5386.865
17,18157,HAJ,FRA,140,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,EDDF,50.033333,8.570556,364.0,5386.865
18,18132,HAJ,AMS,150,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,EHAM,52.308601,4.763890,-11.0,5386.865
19,18197,HAJ,VIE,160,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,LOWW,48.110298,16.569700,600.0,5386.865
20,18162,HAJ,HRG,180,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,HEGN,27.178301,33.799400,52.0,5386.865
21,18184,HAJ,SAW,80,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,LTFJ,40.898602,29.309200,312.0,5386.865
22,18145,HAJ,CDG,140,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,LFPG,49.012798,2.550000,392.0,5386.865
23,18199,HAJ,ZRH,160,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,LSZH,47.464699,8.549170,1416.0,5386.865
24,18180,HAJ,PMI,80,yes,yes,yes,yes,yes,yes,...,Germany,EDDV,52.461102,9.68508,183,LEPA,39.551701,2.738810,27.0,5386.865


## 4. Airport Flight Operations & Movements

Inspect operational flight traffic records, standardize column schemas, remove sparse indicators, aggregate monthly airport flight movements, and provide analytical helper functions.

In [21]:
# Loading the air traffic dataset, containing 2025 and 2026 data
air_traffic = pd.read_csv("./airport traffic.csv")

air_traffic.head()

,YEAR,MONTH_NUM,MONTH_MON,FLT_DATE,APT_ICAO,APT_NAME,STATE_NAME,FLT_DEP_1,FLT_ARR_1,FLT_TOT_1,FLT_DEP_IFR_2,FLT_ARR_IFR_2,FLT_TOT_IFR_2
0,2025,2,FEB,2025-02-01,UDYZ,Yerevan,Armenia,39,37,76,NaN,NaN,NaN
1,2025,2,FEB,2025-02-01,LOWG,Graz,Austria,10,8,18,NaN,NaN,NaN
2,2025,2,FEB,2025-02-01,LOWI,Innsbruck,Austria,74,74,148,NaN,NaN,NaN
3,2025,2,FEB,2025-02-01,LOWK,Klagenfurt,Austria,3,6,9,NaN,NaN,NaN
4,2025,2,FEB,2025-02-01,LOWL,Linz,Austria,13,14,27,NaN,NaN,NaN


In [25]:
# Checking for Null Values
print("\nMissing values per column:")
air_traffic.isna().sum()


Missing values per column:


YEAR                 0
MONTH_NUM            0
MONTH_MON            0
FLT_DATE             0
APT_ICAO             0
APT_NAME             0
STATE_NAME           0
FLT_DEP_1            0
FLT_ARR_1            0
FLT_TOT_1            0
FLT_DEP_IFR_2    83034
FLT_ARR_IFR_2    83034
FLT_TOT_IFR_2    83034
dtype: int64

In [26]:
# Removing the columns with needless data and many null values
air_traffic.drop(columns = ['FLT_DEP_IFR_2','FLT_ARR_IFR_2', 'FLT_TOT_IFR_2'], inplace = True)

# Changing column names
air_traffic.columns = air_traffic.columns.str.lower()

air_traffic.rename(columns = {"apt_icao": "icao_code","apt_name": "airport_city", 
                              "flt_dep_1": "departures", "flt_arr_1": "arrivals", 
                              "flt_tot_1": "total_flights", "state_name": "country"}, inplace = True)
# Final check
air_traffic.head()

,year,month_num,month_mon,flt_date,icao_code,airport_city,country,departures,arrivals,total_flights
0,2025,2,FEB,2025-02-01,UDYZ,Yerevan,Armenia,39,37,76
1,2025,2,FEB,2025-02-01,LOWG,Graz,Austria,10,8,18
2,2025,2,FEB,2025-02-01,LOWI,Innsbruck,Austria,74,74,148
3,2025,2,FEB,2025-02-01,LOWK,Klagenfurt,Austria,3,6,9
4,2025,2,FEB,2025-02-01,LOWL,Linz,Austria,13,14,27


In [27]:
# Group by airport data and month, then sum the flight columns
air_traffic_cleaned = (air_traffic.groupby(["icao_code", "airport_city", "country", "month_num", "month_mon"])
                        [["departures", "arrivals", "total_flights"]].sum().reset_index())

# Export clean operational dataset
air_traffic_cleaned.to_csv("clean_air_traffic_2025.csv", index=False)
air_traffic_cleaned.head(12)

,icao_code,airport_city,country,month_num,month_mon,departures,arrivals,total_flights
0,BIKF,Keflavik,Iceland,1,JAN,1892,1887,3779
1,BIKF,Keflavik,Iceland,2,FEB,1918,1927,3845
2,BIKF,Keflavik,Iceland,3,MAR,2258,2265,4523
3,BIKF,Keflavik,Iceland,4,APR,2567,2574,5141
4,BIKF,Keflavik,Iceland,5,MAY,2928,2927,5855
5,BIKF,Keflavik,Iceland,6,JUN,3339,3356,6695
6,BIKF,Keflavik,Iceland,7,JUL,3685,3680,7365
7,BIKF,Keflavik,Iceland,8,AUG,3814,3823,7637
8,BIKF,Keflavik,Iceland,9,SEP,3032,3028,6060
9,BIKF,Keflavik,Iceland,10,OCT,2451,2435,4886


In [28]:
# Operational Helper Functions

# function to retrieve annual total flight numbers per airport
def annual_flights(df, icao_code):
  return df[df["icao_code"] == icao_code]["total_flights"].sum()

In [29]:
annual_flights(air_traffic, "EDDH")

np.int64(121345)

In [30]:
# function to print annual stats
def annual_stats(df, icao_code):
  airport_df = df[df["icao_code"] == icao_code]

  if airport_df.empty:
    print(f"Airport {icao_code} not found.")
    return None

  annual = airport_df[["departures", "arrivals", "total_flights"]].sum()

  print(f"--- Statistics for {icao_code} ---")
  print("\nAnnual Totals:")
  print(annual)

In [31]:
annual_stats(air_traffic, "EDDH")

--- Statistics for EDDH ---

Annual Totals:
departures        60637
arrivals          60708
total_flights    121345
dtype: int64
